In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b_chir = 7.4515*2*pi
ω_b_ATS = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv_chir = Z_b*ω_b_chir
E_Cb_chir = C_b_inv_chir/2
C_b_inv_ATS = Z_b*ω_b_ATS
E_Cb_ATS = C_b_inv_ATS/2


E_J = 37.12*2*pi*1.5


##Drive


ω_d=2ω_a# Drive frequency
ω_p=2ω_a-ω_d

ϵ_p = 0.62#0.5816119999999999
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 8.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#κ_b/2#3e-3*κ_2 #from Leon's paper

# First change of variables
Q_a, Q_b = 1/κ_a, 1/κ_b
Q_a_tilde, Q_b_tilde_chir, Q_b_tilde_ATS = ω_a/κ_a, ω_b_chir/κ_b, ω_b_ATS/κ_b
β_a, β_b_chir, β_b_ATS= 4E_J*sin(ϵ_p)φ_a^2/(ω_a), 4E_J*sin(ϵ_p)φ_b^2/ω_b_chir, 4E_J*sin(ϵ_p)φ_b^2/ω_b_ATS
λ_a, λ_b = φ_a, φ_b
ϵ_λ_d_chir, ϵ_λ_d_ATS = ϵ_d*2*λ_b/ω_b_chir, ϵ_d*2*λ_b/ω_b_ATS
ν_d = 1;

# Second change of variables
κ_a_tilde_ATS, κ_b_tilde_ATS = 1/(4*Q_a_tilde*sqrt(1-1/(4Q_b_tilde_ATS^2))), 1/(2*Q_b_tilde_ATS*sqrt(1-1/(4Q_b_tilde_ATS^2)))
β_a_tilde_ATS, β_b_tilde_ATS = β_a/(2-1/(2Q_b_tilde_ATS^2)), β_b_ATS/(1-1/(4Q_b_tilde_ATS^2))
ϵ_d_tilde_ATS = ϵ_λ_d_ATS/(1-1/(4Q_b_tilde_ATS^2))
ν_d_tilde_ATS = ν_d/sqrt(1-1/(4Q_b_tilde_ATS^2))
γ_ATS = (1-1/(4Q_a_tilde^2))/(1-1/(4Q_b_tilde_ATS^2))/2;

In [ ]:
g2/2/pi*1e3

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_p*cos(ω_p*t))
    a_a_st = real(a*conj(a))
    J_1 = besselj(1, sqrt(9.5)*sqrt(2Z_a)*sqrt(a_a_st)/ϕ_quantum)
    J_2 = besselj(2, sqrt(9.5)*sqrt(2Z_a)*sqrt(a_a_st)/ϕ_quantum)
    J_3 = besselj(3, sqrt(9.5)*sqrt(2Z_a)*sqrt(a_a_st)/ϕ_quantum)
    coupling_b = -sqrt(Z_b)/sqrt(2)/ϕ_quantum*a^2*J_2/(9.5*a_a_st)
    coupling_a = -sqrt(Z_b)/sqrt(2)/ϕ_quantum*(sqrt(9.5*Z_a)/sqrt(2)/ϕ_quantum*(J_1-J_3)/(9.5*2*a_a_st^(3/2))*(a^3*conj(b)+a_a_st*conj(a)*b)+J_2*(-a^3*conj(b)/a_a_st^2+b/a)/9.5)
    #coupling_a = Z_a*sqrt(Z_b)*sqrt(2)*(-conj(a)*b+6*(a^3*conj(b)+3*a*conj(a)^2*b)/12*Z_a/ϕ_quantum^2)/4/ϕ_quantum^3
    #coupling_b = Z_a*sqrt(Z_b)*sqrt(2)*(-a^2+6*a^3*conj(a)/6*Z_a/ϕ_quantum^2)/8/ϕ_quantum^3
        
    du[1] = -1im*(ω_a*a + sin_pump*coupling_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sin_pump*coupling_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
t_max = 300/κ_b
t_min = 0.0
t_min_save = t_max - 2pi/(ω_a)*2
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)/(100sqrt(2)*3)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)


drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=[-5]#range(-6,6,10)
Re_a_list=[5]#range(-6,6,10)

Final_mode_a_ATS = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_ATS = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_a_ATS_avg = Dict{Tuple{Float64,Float64}, ComplexF64}()
Final_mode_b_ATS_avg = Dict{Tuple{Float64,Float64}, ComplexF64}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j

        system = SystemATS_ladder(ω_a, ω_b_ATS, E_J, E_Ca, E_Cb_ATS, a_0_j, b_0)
        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment);

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
        x_a, p_a = (a+conj(a))/sqrt(2), 1im*(conj(a)-a)/sqrt(2)
        x_b, p_b = (b+conj(b))/sqrt(2), 1im*(conj(b)-b)/sqrt(2)

        x_lam_a, x_lam_b, p_lam_a, p_lam_b = sqrt(2)*λ_a*x_a, sqrt(2)*λ_b*x_b, sqrt(2)*λ_a*p_a, sqrt(2)*λ_b*p_b
        x_a_tilde, x_b_tilde, p_a_tilde, p_b_tilde = x_lam_a, x_lam_b, (p_lam_a+x_lam_a/(2*Q_a_tilde))/sqrt(1-1/(4*Q_b_tilde_ATS^2)), (p_lam_b+x_lam_b/(2*Q_b_tilde_ATS))/sqrt(1-1/(4*Q_b_tilde_ATS^2))
        u_a =@. cos(ω_b_ATS*t_list/2)*x_a_tilde - sin(ω_b_ATS*t_list/2)*p_a_tilde
        v_a =@. sin(ω_b_ATS*t_list/2)*x_a_tilde + cos(ω_b_ATS*t_list/2)*p_a_tilde
        u_b =@. cos(ω_b_ATS*t_list)*x_b_tilde - sin(ω_b_ATS*t_list)*p_b_tilde
        v_b =@. sin(ω_b_ATS*t_list)*x_b_tilde + cos(ω_b_ATS*t_list)*p_b_tilde;

        a_R = (u_a+1im*v_a)/sqrt(2)
        b_R = (u_b+1im*v_b)/sqrt(2)
        Final_mode_a_ATS[(re_i,im_j)] = a_R/sqrt(2)
        Final_mode_b_ATS[(re_i,im_j)] = b_R/sqrt(2)

        Final_mode_a_ATS_avg[(re_i,im_j)] = sum(a_R)/length(a_R)/sqrt(2)
        Final_mode_b_ATS_avg[(re_i,im_j)] = sum(b_R)/length(b_R)/sqrt(2)

    end
end

In [ ]:
#prediction
b_0_norm = 1/(12*(β_a/2+β_b_ATS/2))+sqrt((1/(12*(β_a/2+β_b_ATS/2)))^2+ϵ_λ_d_ATS/3/β_b_ATS)#0.40239616164970887
#a_0_norm = 0.4920305631589086

gamma = 1 + β_b_ATS/β_a
a_1 = -b_0_norm/(2*gamma)#- β_a / 2*(0.4920305631589086^2+0.40239616164970887^2)
a_0_norm = sqrt(-a_1/(β_a/2)-b_0_norm^2)

b_m1_norm = 2β_b_ATS /2*b_0_norm*a_0_norm
b_2 = β_b_ATS * a_1 / β_a

θ_b_0 = -2*b_0_norm/(ϵ_λ_d_ATS/2)*κ_b/ω_b_ATS#-0.11041398179241003
θ_a_0_1 = θ_b_0/2
θ_a_0_2 = θ_b_0/2+π
θ_b_m1 = 3*θ_a_0_1

b_0 = b_0_norm*exp(1im*θ_b_0)
a_0_1 = a_0_norm*exp(1im*θ_a_0_1)
a_0_2 = a_0_norm*exp(1im*θ_a_0_2)
b_m1 = b_m1_norm*exp(1im*θ_b_m1)

b_0_norm_prec = 0.33636495810132455
a_0_norm_prec = 0.47287700802978716

a_1_prec = -0.06895043097048979
b_m1_norm_prec = -0.07201944643601312
b_2_prec = β_b_ATS * a_1_prec / β_a

θ_b_0_prec = -0.141756683610786
θ_a_0_1_prec = θ_b_0_prec/2
θ_a_0_2_prec = θ_b_0_prec/2+π
θ_b_m1_prec = 3*θ_a_0_1_prec

b_0_prec = b_0_norm_prec*exp(1im*θ_b_0_prec)
a_0_1_prec = a_0_norm_prec*exp(1im*θ_a_0_1_prec)
a_0_2_prec = a_0_norm_prec*exp(1im*θ_a_0_2_prec)

b_0_cat_state = 0
a_0_cat_state_1 = 1im*sqrt(ϵ_d*λ_b/ω_b_ATS/(β_b_ATS/2))
a_0_cat_state_2 = -1im*sqrt(ϵ_d*λ_b/ω_b_ATS/(β_b_ATS/2));


In [ ]:
p0 = plot()

first_ATS     = true
first_ATS_avg = true
first_chiral  = true

# ATS (red dots) + ATS avg (red crosses)

for key in keys(Final_mode_a_ATS)
    z = Final_mode_b_ATS[key]
    z_avg = Final_mode_b_ATS_avg[key]   # should be a single complex number

    scatter!(
        p0,
        real.(z),
        imag.(z),
        ms = 1,
        color = :red,
        label = first_ATS ? "ATS" : "",
    )
    first_ATS = false
end
display(p0)

In [ ]:
p0 = plot()

first_ATS     = true
first_ATS_avg = true
first_chiral  = true

# ATS (red dots) + ATS avg (red crosses)

for key in keys(Final_mode_a_ATS)
    z = Final_mode_a_ATS[key]
    z_avg = Final_mode_a_ATS_avg[key]   # should be a single complex number

    scatter!(
        p0,
        real.(z),
        imag.(z),
        ms = 1,
        color = :red,
        label = first_ATS ? "ATS" : "",
    )
    first_ATS = false
end
for key in keys(Final_mode_a_ATS)
    z = Final_mode_a_ATS[key]
    z_avg = Final_mode_a_ATS_avg[key]   # should be a single complex number

    scatter!(
        p0,
        [real(z_avg)],
        [imag(z_avg)],
        ms = 1,
        marker = :x,
        color = :black,
        label = first_ATS_avg ? "ATS avg" : "",
    )
    first_ATS_avg = false
end



# Implicit curve
f(x, p) = (x^2 + p^2)^2 + 4ϵ_λ_d_ATS/β_b_ATS*(x^2 - p^2)

x_range = range(-4, 4; length=400)
p_range = range(-4, 4; length=400)

contour!(
    p0,
    x_range,
    p_range,
    (x, p) -> f(x, p),
    levels = [0.0],
    linewidth = 2,
    linestyle = :dash,
    color = :blue,
    label = "(x²+p²)² = 4(x²-p²)",
)



# Predictions
scatter!(p0, [real(a_0_1)], [imag(a_0_1)], ms = 2, label = "", color=Perso_palette_hex[3])
θ = range(0, 2π; length=300)
plot!(
    p0,
    real(a_0_1) .+ a_1 .* cos.(-θ),
    imag(a_0_1) .+ a_1 .* sin.(-θ),
    linestyle = :dash,
    color = Perso_palette_hex[3],
    label = "",
)
scatter!(p0, [real(a_0_2)], [imag(a_0_2)], ms = 2, label = "", color=Perso_palette_hex[3])
plot!(
    p0,
    real(a_0_2) .+ a_1 .* cos.(-θ),
    imag(a_0_2) .+ a_1 .* sin.(-θ),
    linestyle = :dash,
    color = Perso_palette_hex[3],
    label = "",
)
scatter!(p0, [real(a_0_cat_state_1)], [imag(a_0_cat_state_1)], ms = 2, label = "Approximation", color=Perso_palette_hex[3])
scatter!(p0, [real(a_0_cat_state_2)], [imag(a_0_cat_state_2)], ms = 2, label = "", color=Perso_palette_hex[3])
scatter!(p0, [real(a_0_1_prec)], [imag(a_0_1_prec)], ms = 2, label = "Refined Approximation", color=Perso_palette_hex[4])
plot!(
    p0,
    real(a_0_1_prec) .+ a_1_prec .* cos.(-θ),
    imag(a_0_1_prec) .+ a_1_prec .* sin.(-θ),
    linestyle = :dash,
    color = Perso_palette_hex[4],
    label = "",
)
scatter!(p0, [real(a_0_2_prec)], [imag(a_0_2_prec)], ms = 2, label = "", color=Perso_palette_hex[4])
plot!(
    p0,
    real(a_0_2_prec) .+ a_1_prec .* cos.(-θ),
    imag(a_0_2_prec) .+ a_1_prec .* sin.(-θ),
    linestyle = :dash,
    color = Perso_palette_hex[4],
    label = "",
)

# Chiral (black dots)
#for key in keys(Final_mode_a_chir)
#    z = Final_mode_a_chir[key]

#    scatter!(
#        p0,
#        real.(z),
#        imag.(z),
#        ms = 1,
#        color = :black,
#        label = first_chiral ? "Chiral " : "",
#    )
#    first_chiral = false
#end

# Labels and title
plot!(
    p0,
    xlabel = L"\Re(\tilde a_R)",
    ylabel = L"\Im(\tilde a_R)",
    title  = L"g_2/2\pi=" * string(round(g2/2pi*1000, digits=3)) * " MHz",
    top_margin = 5mm,
    aspect_ratio = 1,
    xlims=(-1,1),
    ylims=(-1,1),
)
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/satellites/prediction_satellites_memory_30_01_2026_cat_only.pdf")

In [ ]:
p0 = plot()

first_ATS     = true
first_ATS_avg = true
first_chiral  = true

# ATS (red dots) + ATS avg (red crosses)
for key in keys(Final_mode_b_ATS)
    z = Final_mode_b_ATS[key]
    z_avg = Final_mode_b_ATS_avg[key]   # should be a single complex number

    scatter!(
        p0,
        real.(z),
        imag.(z),
        ms = 1,
        color = :red,
        label = first_ATS ? "ATS" : "",
    )
    first_ATS = false

    scatter!(
        p0,
        [real(z_avg)],
        [imag(z_avg)],
        ms = 1,
        marker = :x,
        color = :red,
        label = first_ATS_avg ? "ATS avg" : "",
    )
    first_ATS_avg = false
end
# Predictions

scatter!(p0, [real(b_0)], [imag(b_0)], ms = 2, label = "", color=Perso_palette_hex[3])
θ = range(0, 2π; length=300)
plot!(
    p0,
    real(b_0) .- b_m1_norm .*cos.(-θ.-θ_b_m1) .+ b_2 .* cos.(-2 .*θ),
    imag(b_0) .+ b_m1_norm .*sin.(-θ.-θ_b_m1) .+ b_2 .* sin.(-2 .*θ),
    linestyle = :dash,
    color = Perso_palette_hex[3],
    label = "",
)
scatter!(p0, [real(b_0_cat_state)], [imag(b_0_cat_state)], ms = 2, label = "Approximation", color=Perso_palette_hex[3])
scatter!(p0, [real(b_0_prec)], [imag(b_0_prec)], ms = 2, label = "Refined Approximation", color=Perso_palette_hex[4])
plot!(
    p0,
    real(b_0_prec) .- b_m1_norm_prec .*cos.(-θ.-θ_b_m1_prec) .+ b_2_prec .* cos.(-2 .*θ),
    imag(b_0_prec) .+ b_m1_norm_prec .*sin.(-θ.-θ_b_m1_prec) .+ b_2_prec .* sin.(-2 .*θ),
    linestyle = :dash,
    color = Perso_palette_hex[4],
    label = "",
)

# Chiral (black dots)
#for key in keys(Final_mode_b_chir)
#    z = Final_mode_b_chir[key]

#    scatter!(
#        p0,
#        real.(z),
#        imag.(z),
#        ms = 1,
#        color = :black,
#        label = first_chiral ? "Chiral" : "",
#    )
#    first_chiral = false
#end

# Labels and title
plot!(
    p0,
    xlabel = L"\Re(\tilde b_R)",
    ylabel = L"\Im(\tilde b_R)",
    title  = L"g_2/2\pi=" * string(round(g2/2pi*1000, digits=3)) * " MHz",
    top_margin = 5mm,
    aspect_ratio = 1,
    legend=:bottomleft,
    #xlims=(-.5,.75),
    #ylims=(-.4,.25),
)
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/satellites/prediction_satellites_buffer_30_01_2026_cat_only.pdf")


In [ ]:
2E_J*sin(0.58)φ_b^2/ω_b_ATS

In [ ]:
eps_p = 0.4:0.01:0.5*pi
beta_a =@. 2E_J*sin(eps_p)*φ_a^2/(ω_a)
beta_b =@. 2E_J*sin(eps_p)*φ_b^2/ω_b_ATS

eps = 0.009779030042836782

b =@. 1/(12*(beta_a+beta_b))+sqrt((1/(12*(beta_a+beta_b)))^2+eps/(3*beta_b))
gamma =@. 1+beta_b/beta_a
gamma_p =@. 1+2*beta_b/(3*beta_a)

expr =@. (
    (1/(2*beta_a*gamma_p) - b)^2 * (1/(4*beta_b) - b)^2
    - 2*(1/(2*beta_a*gamma_p) - b)*(1/(4*beta_b) - b) * b/(2*beta_a*gamma)
    - (1/(4*beta_b) - b)^2 * (-2*b^2 + eps/beta_b)
    + 4*b*(1/(4*beta_b) - b) * (-2*b^2 + eps/beta_b)
    + (-3*b^2 + eps/beta_b)^2
);

In [ ]:
plot(eps_p, expr)